# BrainAI on Colab (GPU)
Runtime > Change runtime type > T4 GPU. Everything runs on the session's local disk; download the checkpoint zip at the end.

In [ ]:
# 1) repo + dependencies (no Google login needed)
!apt-get -qq install git-lfs && git lfs install
!git clone https://github.com/ZahidServers/BrainAI.git /content/BrainAI
%cd /content/BrainAI
!pip -q install torch datasets pillow opencv-python-headless transformers accelerate

In [ ]:
# 2) storage: this session's local disk (no Google Drive needed)
import os
D = '/content/brainai'
os.makedirs(D + '/data', exist_ok=True)
print('storage:', D)

## 1. Text corpora (reading/writing)

In [ ]:
!python tools/fetch_text.py --out $D/data --mb 500
!ln -sf $D/data/*.txt data/   # corpus.py auto-discovers every data/*.txt as a domain

## 2. Sensor data (images, audio, video)

In [ ]:
!python tools/fetch_sensors.py --out $D/sensors --n 20000
!ln -sfn $D/sensors sensors

## 2b. Robot data (state, actions, camera) - LeRobot format

In [ ]:
!python tools/fetch_robot.py --out $D/robot_data --skip koch_pick_place_1_lego
!python tools/fetch_robot.py --out $D/robot_data --humanoid          # Unitree H1, ~1.4 GB
# optional, 9-18 GB each: !python tools/fetch_robot.py --out $D/robot_data --humanoid --humanoid-big
!ln -sfn $D/robot_data robot_data

## 2c. Train the perception encoders (held-out accuracy is printed vs chance)

In [ ]:
for m,s in [('images',4000),('audio',3000),('video',2000)]:
    !python train_sensors.py --modality $m --steps $s --batch 64 --scale small

## 2d. Behaviour cloning (state -> action); compares to copy-state baseline

In [ ]:
import glob
for d in sorted(glob.glob('robot_data/*')):
    !python train_robot.py --dataset $d --steps 3000 --scale small

## 3. Distil Qwen into BrainAI (text; teacher writes, student trains)
Byte tokenizer != Qwen BPE, so logit-KL is impossible; we use sequence distillation.

In [ ]:
!python distill.py generate --teacher Qwen/Qwen2.5-1.5B-Instruct --samples 50000


## 4. Train (text + distilled). Start with `small`, move to `base` once loss curves look sane.

In [ ]:
!python train_multidomain.py --scale small --steps 40000 --batch 32 --seq 256

In [ ]:
# Local disk is wiped when the session ends: download the trained weights now.
!zip -qr brainai_checkpoints.zip checkpoints
from google.colab import files
files.download('brainai_checkpoints.zip')